# Hyperspectral Sampler

Currently, the 3DGS model does not support hyperspectral CT datasets. However, we can grab a small subset of energy channels from a hyperspectral dataset in order to simulate a dual-energy and/or multi-energy dataset.

The resulting dataset can then be used as input for the dataset_preprocess step

In [174]:
import scipy.io as sio
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image
import h5py
from tomopy.prep.stripe import remove_stripe_based_sorting

In [175]:
# # --------- Lizard dataset ---------
# SCAN_PARAM_PATH = '../../data_raw/lizard/Lizard_head_scan_parameters.txt'

# energy_axis = sio.loadmat('../../data_raw/lizard/Energy_axis.mat')
# energy_axis = energy_axis['E_axis']
# print("Energy axis:", energy_axis.shape)

# data = sio.loadmat('../../data_raw/lizard/lizard_180Proj_noSupp_1_180.mat')
# data = data['lizard_180Proj_noSupp_1_180']
# print("Data:", data.shape)
# print("Data shape = [vertical pixels, angles, horizontal pixels, energy channels]")

# # FF = sio.loadmat('../../data_raw/lizard/FF.mat')
# # FF = FF['FF_lizard'].astype(np.float32)
# # print("FF shape:", FF.shape) 

# # data = sio.loadmat('../../data_raw/lizard/lizard_180Proj_Supp_1_180.mat')
# # data = data['lizard_180Proj_Supp_1_180']
# # print("Data:", data.shape)
# # print("Data shape = [vertical pixels, angles, horizontal pixels, energy channels]")

# OUTPUT_DIR = 'output/lizard_temp'
# ENERGY_WINDOWS = [
#     (73, 78),
#     (123, 128),
#     (160, 180),
# ]

In [176]:
# # --------- Mouse dataset (hindlimb) ---------
# SCAN_PARAM_PATH = '../../data_raw/mouse/Hindlimb_scan_parameters.txt'

# energy_axis = sio.loadmat('../../data_raw/mouse/E_axis.mat')
# energy_axis = energy_axis['E_axis']
# print("Energy axis:", energy_axis.shape)

# data = None
# with h5py.File('../../data_raw/mouse/DS_Mouse_hindlimb_sinogram.mat', 'r') as f:
#     data = f['SS'][:]
#     data = np.transpose(data, (3, 2, 1, 0)) # convert to lizard dataset shape (standard)
# print("Data:", data.shape)
# print("Data shape = [vertical pixels, angles, horizontal pixels, energy channels]")

# OUTPUT_DIR = 'output/mouse_hindlimb_temp'
# ENERGY_WINDOWS = [
#     (73, 78),
#     (123, 128),
#     (170, 190),
# ]

In [ ]:
# # --------- Mouse dataset (forelimb) ---------
# SCAN_PARAM_PATH = '../../data_raw/mouse/Forelimb_scan_parameters.txt'

# energy_axis = sio.loadmat('../../data_raw/mouse/E_axis.mat')
# energy_axis = energy_axis['E_axis']
# print("Energy axis:", energy_axis.shape)

# data = None
# with h5py.File('../../data_raw/mouse/TS_Mouse_forelimb_sinogram.mat', 'r') as f:
#     data = f['SS'][:]
#     data = np.transpose(data, (3, 2, 1, 0)) # convert to lizard dataset shape (standard)
# print("Data:", data.shape)
# print("Data shape = [vertical pixels, angles, horizontal pixels, energy channels]")

# OUTPUT_DIR = 'output/mouse_forelimb_temp'
# ENERGY_WINDOWS = [
#     (73, 78),
#     (123, 128),
#     (170, 190),
# ]

Energy axis: (1, 400)
Data: (80, 180, 80, 400)
Data shape = [vertical pixels, angles, horizontal pixels, energy channels]


In [ ]:
# --------- Iodine phantom dataset ---------
SCAN_PARAM_PATH = '../../data_raw/iodine_phantom/Iodine_Phantom_scan_parameters.txt'

energy_axis = sio.loadmat('../../data_raw/iodine_phantom/E_axis.mat')
energy_axis = energy_axis['E_axis']
print("Energy axis:", energy_axis.shape)

data = None
with h5py.File('../../data_raw/iodine_phantom/Iodine_phantom_sinogram.mat', 'r') as f:
    data = f['SSC'][:]
print("Data:", data.shape)
print("Data shape = [vertical pixels, angles, horizontal pixels, energy channels]")

OUTPUT_DIR = 'output/iodine_phantom_temp'
ENERGY_WINDOWS = [
    (73, 78),
    (123, 128),
    (170, 190),
]

In [179]:
NUM_ANGLES = data.shape[1]
NUM_CHANNELS = data.shape[3]
channel_windows = [(int(start), int(stop)) for start, stop in ENERGY_WINDOWS]

DESTRIPE_SORT_SIZE = 11
WINDOW_REDUCTION = 'mean'

for start, stop in channel_windows:
    if start < 0 or stop > NUM_CHANNELS or start >= stop:
        raise ValueError(f"Invalid energy window: ({start}, {stop})")

CHANNEL_INDICES = [int((start + stop - 1) // 2) for start, stop in channel_windows]
ENERGY_CHANNELS = [float(np.mean(energy_axis[0, start:stop])) for start, stop in channel_windows]

energies = []
for i, ((start, stop), center_index, energy) in enumerate(zip(channel_windows, CHANNEL_INDICES, ENERGY_CHANNELS)):
    print(
        f"Output channel {i}: window [{start}, {stop}) centered near detector channel {center_index}, "
        f"effective energy {energy:.4f} keV"
    )
    energies.append(energy)
print(energies)

Output channel 0: window [73, 78) centered near detector channel 75, effective energy 21.9067 keV
Output channel 1: window [123, 128) centered near detector channel 125, effective energy 35.8919 keV
Output channel 2: window [170, 190) centered near detector channel 179, effective energy 51.1356 keV
[21.9067197198409, 35.89185291276528, 51.13564809305285]


In [180]:
with open(SCAN_PARAM_PATH, 'r') as f:
    scan_parameters_raw = f.read()

    scan_parameters_lines = []
    for line in scan_parameters_raw.splitlines():
        line = line.split('#', 1)[0].strip()
        if not line:
            continue
        scan_parameters_lines.append(line)

scan_parameters = "\n".join(scan_parameters_lines)

In [181]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

for i, ((window_start, window_stop), center_index, effective_energy) in enumerate(zip(channel_windows, CHANNEL_INDICES, ENERGY_CHANNELS)):
    OUTPUT_DIR_CHANNEL = os.path.join(OUTPUT_DIR, f"{i}")
    os.makedirs(OUTPUT_DIR_CHANNEL, exist_ok=True)

    config_path = os.path.join(OUTPUT_DIR_CHANNEL, "config.txt")
    with open(config_path, "w") as f:
        f.write(scan_parameters)
        f.write("\n")

    # Integrate an explicit energy window to increase photon statistics before destriping.
    channel_stack = data[:, :, :, window_start:window_stop].astype(np.float32)
    if WINDOW_REDUCTION == 'sum':
        channel_stack = np.sum(channel_stack, axis=3)
    else: # WINDOW_REDUCTION == 'mean'
        channel_stack = np.mean(channel_stack, axis=3)

    # Build full energy-channel stack in sinogram space: [row, angle, col]
    tomo = np.transpose(channel_stack, (1, 0, 2)).astype(np.float32)
    tomo = np.nan_to_num(tomo, nan=0.0, posinf=0.0, neginf=0.0)

    tomo_corr = remove_stripe_based_sorting(tomo, size=DESTRIPE_SORT_SIZE, dim=1)

    tomo_corr = np.nan_to_num(tomo_corr, nan=0.0, posinf=0.0, neginf=0.0)
    channel_stack_corr = np.transpose(tomo_corr, (1, 0, 2)).astype(np.float32)

    for angle_idx in range(NUM_ANGLES):
        projection = channel_stack_corr[:, angle_idx, :]
        filename = f"{angle_idx + 1:04d}.tif"
        filepath = os.path.join(OUTPUT_DIR_CHANNEL, filename)
        Image.fromarray(projection.astype(np.float32)).save(filepath)

    print(f"All 180 destriped projections saved to {OUTPUT_DIR_CHANNEL}")
    print(
        f"Using energy window [{window_start}, {window_stop}) centered near detector channel {center_index} "
        f"-> effective energy {effective_energy:.4f} keV"
    )

All 180 destriped projections saved to output/mouse_forelimb_temp/0
Using energy window [73, 78) centered near detector channel 75 -> effective energy 21.9067 keV
All 180 destriped projections saved to output/mouse_forelimb_temp/1
Using energy window [123, 128) centered near detector channel 125 -> effective energy 35.8919 keV
All 180 destriped projections saved to output/mouse_forelimb_temp/2
Using energy window [170, 190) centered near detector channel 179 -> effective energy 51.1356 keV
